In [1]:

from pathlib import Path

root = Path("/kaggle/input/datasets/deepeshghimire/advance-project")

print("Root exists:", root.exists())
print("Root path:", root)

if root.exists():
    for p in sorted(root.rglob("*")):
        if p.is_dir():
            print("DIR :", p)
        else:
            print("FILE:", p, f"| {p.stat().st_size:,} bytes")


Root exists: True
Root path: /kaggle/input/datasets/deepeshghimire/advance-project
FILE: /kaggle/input/datasets/deepeshghimire/advance-project/._Task07_Pancreas | 120 bytes
DIR : /kaggle/input/datasets/deepeshghimire/advance-project/Task07_Pancreas
FILE: /kaggle/input/datasets/deepeshghimire/advance-project/Task07_Pancreas/._dataset.json | 176 bytes
FILE: /kaggle/input/datasets/deepeshghimire/advance-project/Task07_Pancreas/._imagesTr | 120 bytes
FILE: /kaggle/input/datasets/deepeshghimire/advance-project/Task07_Pancreas/._imagesTs | 120 bytes
FILE: /kaggle/input/datasets/deepeshghimire/advance-project/Task07_Pancreas/._labelsTr | 120 bytes
FILE: /kaggle/input/datasets/deepeshghimire/advance-project/Task07_Pancreas/dataset.json | 28,581 bytes
DIR : /kaggle/input/datasets/deepeshghimire/advance-project/Task07_Pancreas/imagesTr
FILE: /kaggle/input/datasets/deepeshghimire/advance-project/Task07_Pancreas/imagesTr/._pancreas_001.nii.gz | 176 bytes
FILE: /kaggle/input/datasets/deepeshghimire

In [ ]:
!pip install -q monai nibabel scikit-learn

In [2]:

from pathlib import Path
import json

root = Path(
    "/kaggle/input/datasets/deepeshghimire/"
    "advance-project/Task07_Pancreas"
)

def real_files(folder):
    return sorted(
        p for p in folder.iterdir()
        if p.is_file()
        and not p.name.startswith("._")
        and p.name.endswith((".nii", ".nii.gz"))
    )

images_tr = real_files(root / "imagesTr")
images_ts = real_files(root / "imagesTs")
labels_tr = real_files(root / "labelsTr")

print("Training images:", len(images_tr))
print("Test images:", len(images_ts))
print("Training masks:", len(labels_tr))

print("\nFirst 10 training images:")
for p in images_tr[:10]:
    print(p.name)

print("\nFirst 10 training masks:")
for p in labels_tr[:10]:
    print(p.name)

with open(root / "dataset.json", "r") as f:
    metadata = json.load(f)

print("\nDataset metadata keys:", list(metadata.keys()))
print("\nLabels:", metadata.get("labels"))
print("Image format:", metadata.get("file_ending"))
print("Training entry example:", metadata.get("training", [])[:2])


Training images: 281
Test images: 139
Training masks: 281

First 10 training images:
pancreas_001.nii
pancreas_004.nii
pancreas_005.nii
pancreas_006.nii
pancreas_010.nii
pancreas_012.nii
pancreas_015.nii
pancreas_016.nii
pancreas_018.nii
pancreas_019.nii

First 10 training masks:
pancreas_001.nii
pancreas_004.nii
pancreas_005.nii
pancreas_006.nii
pancreas_010.nii
pancreas_012.nii
pancreas_015.nii
pancreas_016.nii
pancreas_018.nii
pancreas_019.nii

Dataset metadata keys: ['name', 'description', 'reference', 'licence', 'relase', 'tensorImageSize', 'modality', 'labels', 'numTraining', 'numTest', 'training', 'test']

Labels: {'0': 'background', '1': 'pancreas', '2': 'cancer'}
Image format: None
Training entry example: [{'image': './imagesTr/pancreas_290.nii.gz', 'label': './labelsTr/pancreas_290.nii.gz'}, {'image': './imagesTr/pancreas_127.nii.gz', 'label': './labelsTr/pancreas_127.nii.gz'}]


In [3]:

from pathlib import Path
import numpy as np
import nibabel as nib

root = Path(
    "/kaggle/input/datasets/deepeshghimire/"
    "advance-project/Task07_Pancreas"
)

def get_nii_files(folder):
    return {
        p.name: p
        for p in folder.iterdir()
        if p.is_file()
        and not p.name.startswith("._")
        and p.name.endswith(".nii")
    }

images = get_nii_files(root / "imagesTr")
masks = get_nii_files(root / "labelsTr")

print("Images:", len(images))
print("Masks:", len(masks))

missing_masks = sorted(set(images) - set(masks))
missing_images = sorted(set(masks) - set(images))

print("Images without masks:", missing_masks[:10])
print("Masks without images:", missing_images[:10])

shape_mismatches = []
all_mask_labels = set()

for name in sorted(set(images) & set(masks)):
    img = nib.load(str(images[name]))
    mask = nib.load(str(masks[name]))

    if img.shape != mask.shape:
        shape_mismatches.append(
            (name, img.shape, mask.shape)
        )

    values = np.unique(
        np.asanyarray(mask.dataobj)
    )
    all_mask_labels.update(values.tolist())

print("\nShape mismatches:", len(shape_mismatches))
print("Examples:", shape_mismatches[:5])
print("Observed mask labels:", sorted(all_mask_labels))

if images:
    sample_name = sorted(images)[0]
    sample = nib.load(str(images[sample_name]))

    print("\nSample:", sample_name)
    print("Shape:", sample.shape)
    print("Voxel spacing:", sample.header.get_zooms())
    print("Orientation:", nib.aff2axcodes(sample.affine))


Images: 281
Masks: 281
Images without masks: []
Masks without images: []

Shape mismatches: 0
Examples: []
Observed mask labels: [0, 1, 2]

Sample: pancreas_001.nii
Shape: (512, 512, 110)
Voxel spacing: (np.float32(0.644531), np.float32(0.644531), np.float32(2.5))
Orientation: ('R', 'A', 'S')


In [4]:

from pathlib import Path
import numpy as np
import nibabel as nib

root = Path(
    "/kaggle/input/datasets/deepeshghimire/"
    "advance-project/Task07_Pancreas"
)

images_dir = root / "imagesTr"
masks_dir = root / "labelsTr"

shape_errors = []
mask_labels = set()

for image_path in sorted(images_dir.glob("*.nii")):
    if image_path.name.startswith("._"):
        continue

    mask_path = masks_dir / image_path.name

    if not mask_path.exists():
        continue

    image = nib.load(str(image_path))
    mask = nib.load(str(mask_path))

    if image.shape != mask.shape:
        shape_errors.append(
            (image_path.name, image.shape, mask.shape)
        )

    labels = np.unique(np.asanyarray(mask.dataobj))
    mask_labels.update(labels.tolist())

print("Shape mismatch count:", len(shape_errors))
print("Shape mismatch examples:", shape_errors[:5])
print("Unique mask labels:", sorted(mask_labels))


Shape mismatch count: 0
Shape mismatch examples: []
Unique mask labels: [0, 1, 2]


In [6]:
!pip install -q monai nibabel scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 33.9 MB/s eta 0:00:0000:01


In [7]:

from pathlib import Path
import json
import numpy as np
import torch
import monai

from sklearn.model_selection import train_test_split
from monai.data import Dataset, DataLoader, list_data_collate
from monai.transforms import (
    Compose,
    LoadImaged,
    EnsureChannelFirstd,
    Orientationd,
    Spacingd,
    ScaleIntensityRanged,
    CropForegroundd,
    SpatialPadd,
    RandCropByLabelClassesd,
    RandFlipd,
    RandRotate90d,
    EnsureTyped,
    MapTransform,
)
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

ROOT = Path(
    "/kaggle/input/datasets/deepeshghimire/"
    "advance-project/Task07_Pancreas"
)

images_dir = ROOT / "imagesTr"
labels_dir = ROOT / "labelsTr"
OUTPUT_DIR = Path("/kaggle/working/pancreas_model")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Ignore macOS metadata files starting with ._
image_files = sorted(
    p for p in images_dir.glob("*.nii")
    if not p.name.startswith("._")
)

records = []
for image_path in image_files:
    mask_path = labels_dir / image_path.name
    if mask_path.exists():
        records.append({
            "image": str(image_path),
            "label": str(mask_path),
            "patient_id": image_path.stem
        })

assert len(records) > 0, "No image-mask pairs found."

train_records, val_records = train_test_split(
    records,
    test_size=0.15,
    random_state=SEED
)

print("Total pairs:", len(records))
print("Training:", len(train_records))
print("Validation:", len(val_records))
print("Device:", "CUDA GPU" if torch.cuda.is_available() else "CPU")

with open(ROOT / "dataset.json") as f:
    metadata = json.load(f)

print("Dataset classes:", metadata["labels"])


Total pairs: 281
Training: 238
Validation: 43
Device: CUDA GPU
Dataset classes: {'0': 'background', '1': 'pancreas', '2': 'cancer'}


In [8]:
from monai.transforms import MapTransform
class IntegerLabeld(MapTransform):
    """Ensure segmentation labels are integer class IDs."""
    def __call__(self, data):
        d = dict(data)
        d["label"] = d["label"].round().long()
        return d


PATCH_SIZE = (64, 64, 64)

train_transform = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    Orientationd(keys=["image", "label"], axcodes="RAS"),
    ScaleIntensityRanged(
        keys=["image"],
        a_min=-100,
        a_max=300,
        b_min=0.0,
        b_max=1.0,
        clip=True
    ),
    IntegerLabeld(keys=["label"]),
    CropForegroundd(
        keys=["image", "label"],
        source_key="image"
    ),
    SpatialPadd(
        keys=["image", "label"],
        spatial_size=PATCH_SIZE,
        mode="constant"
    ),
    RandCropByLabelClassesd(
        keys=["image", "label"],
        label_key="label",
        spatial_size=PATCH_SIZE,
        num_classes=3,
        ratios=[1, 1, 3],
        num_samples=2,
        allow_smaller=False
    ),
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=0),
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=1),
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=2),
    RandRotate90d(keys=["image", "label"], prob=0.5, max_k=3),
    EnsureTyped(keys=["image"], dtype=torch.float32),
    EnsureTyped(keys=["label"], dtype=torch.long)
])

val_transform = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    Orientationd(keys=["image", "label"], axcodes="RAS"),
    ScaleIntensityRanged(
        keys=["image"],
        a_min=-100,
        a_max=300,
        b_min=0.0,
        b_max=1.0,
        clip=True
    ),
    IntegerLabeld(keys=["label"]),
    CropForegroundd(keys=["image", "label"], source_key="image"),
    SpatialPadd(
        keys=["image", "label"],
        spatial_size=PATCH_SIZE,
        mode="constant"
    ),
    EnsureTyped(keys=["image"], dtype=torch.float32),
    EnsureTyped(keys=["label"], dtype=torch.long)
])

train_ds = Dataset(train_records, train_transform)
val_ds = Dataset(val_records, val_transform)

train_loader = DataLoader(
    train_ds,
    batch_size=1,
    shuffle=True,
    num_workers=0,
    collate_fn=list_data_collate,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_ds,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

print("Data loaders created.")


nvImageCodec is not available; skipping pydicom decoder plugin registration.
/usr/local/lib/python3.13/dist-packages/monai/data/image_reader.py:1166: UserWarning: NvImgCodecPydicomReader: nvImageCodec decoder plugin did not register successfully. Falling back to default pydicom decoders.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/monai/utils/deprecate_utils.py:320: FutureWarning: monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.
  warn_deprecated(argname, msg, warning_category)
nvImageCodec is not available; skipping pydicom decoder plugin registration.


Data loaders created.


In [9]:

from monai.networks.nets import UNet
from monai.losses import DiceCELoss
from monai.metrics import DiceMetric

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = UNet(
    spatial_dims=3,
    in_channels=1,
    out_channels=3,
    channels=(16, 32, 64, 128),
    strides=(2, 2, 2),
    num_res_units=2
).to(device)

loss_function = DiceCELoss(
    to_onehot_y=True,
    softmax=True,
    include_background=False
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-5
)

dice_metric = DiceMetric(
    include_background=False,
    reduction="mean",
    get_not_nans=True
)

print("3D U-Net ready.")
print("Parameters:", sum(p.numel() for p in model.parameters()))


3D U-Net ready.
Parameters: 1187766


In [ ]:
from monai.inferers import sliding_window_inference
from monai.data import decollate_batch
from monai.transforms import AsDiscrete
import gc

MAX_EPOCHS = 50
VAL_EVERY = 1
best_dice = -1.0

post_pred = AsDiscrete(argmax=True, to_onehot=3)
post_label = AsDiscrete(to_onehot=3)

best_model_path = OUTPUT_DIR / "best_pancreas_unet.pth"

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    steps = 0

    for batch in train_loader:
        images = batch["image"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = loss_function(logits, labels)

        loss.backward()
        optimizer.step()

        epoch_loss += float(loss.item())
        steps += 1

        del images, labels, logits, loss

    train_loss = epoch_loss / max(steps, 1)

    if epoch % VAL_EVERY == 0:
        model.eval()
        dice_metric.reset()

        with torch.no_grad():
            for batch in val_loader:
                images = batch["image"].to(device)
                labels = batch["label"].to(device)

                logits = sliding_window_inference(
                    inputs=images,
                    roi_size=PATCH_SIZE,
                    sw_batch_size=1,
                    predictor=model,
                    overlap=0.25
                )

                preds = [post_pred(x) for x in decollate_batch(logits)]
                targets = [post_label(x) for x in decollate_batch(labels)]
                dice_metric(y_pred=preds, y=targets)

                del images, labels, logits, preds, targets

        metric_value = dice_metric.aggregate()
        # Handle MONAI versions that return a tuple
        if isinstance(metric_value, tuple):
            metric_value = metric_value[0]
        if torch.is_tensor(metric_value):
            mean_dice = float(metric_value.mean().item())
        else:
            mean_dice = float(metric_value)

    
        dice_metric.reset()

        print(
            f"Epoch {epoch:03d}/{MAX_EPOCHS} | "
            f"Train loss: {train_loss:.4f} | "
            f"Validation mean Dice: {mean_dice:.4f}"
        )

        if np.isfinite(mean_dice) and mean_dice > best_dice:
            best_dice = mean_dice
            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_mean_dice": best_dice,
                "labels": metadata["labels"],
                "patch_size": PATCH_SIZE
            }, best_model_path)
            print("Saved best checkpoint:", best_model_path)

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

print("\nTraining complete.")
print("Best validation mean Dice:", best_dice)
print("Checkpoint exists:", best_model_path.exists())


Epoch 001/50 | Train loss: 1.7905 | Validation mean Dice: 0.0078
Saved best checkpoint: /kaggle/working/pancreas_model/best_pancreas_unet.pth
Epoch 002/50 | Train loss: 1.3455 | Validation mean Dice: 0.0121
Saved best checkpoint: /kaggle/working/pancreas_model/best_pancreas_unet.pth
Epoch 003/50 | Train loss: 1.1797 | Validation mean Dice: 0.0124
Saved best checkpoint: /kaggle/working/pancreas_model/best_pancreas_unet.pth
Epoch 004/50 | Train loss: 1.1102 | Validation mean Dice: 0.0135
Saved best checkpoint: /kaggle/working/pancreas_model/best_pancreas_unet.pth
Epoch 005/50 | Train loss: 1.0872 | Validation mean Dice: 0.0156
Saved best checkpoint: /kaggle/working/pancreas_model/best_pancreas_unet.pth
Epoch 006/50 | Train loss: 1.0511 | Validation mean Dice: 0.0191
Saved best checkpoint: /kaggle/working/pancreas_model/best_pancreas_unet.pth
Epoch 007/50 | Train loss: 1.0287 | Validation mean Dice: 0.0178
Epoch 008/50 | Train loss: 1.0074 | Validation mean Dice: 0.0289
Saved best checkpo

In [ ]:

# ============================================
# 3D U-Net Pancreas Segmentation Visualization
# ============================================

import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from monai.inferers import sliding_window_inference

# 1. Find the saved model
possible_paths = [
    Path("/kaggle/working/pancreas_model/best_pancreas_unet.pth"),
    Path("/kaggle/working/best_pancreas_unet.pth"),
]

MODEL_PATH = next((p for p in possible_paths if p.exists()), None)

if MODEL_PATH is None:
    for root, dirs, files in os.walk("/kaggle/working"):
        if "best_pancreas_unet.pth" in files:
            MODEL_PATH = Path(root) / "best_pancreas_unet.pth"
            break

if MODEL_PATH is None:
    raise FileNotFoundError(
        "Model file not found. Check the training cell and saved checkpoint path."
    )

print("Model found:", MODEL_PATH)

# 2. Check required training variables
required_vars = ["model", "device", "val_ds", "PATCH_SIZE"]

missing = [name for name in required_vars if name not in globals()]
if missing:
    raise NameError(
        f"Missing variables: {missing}. Run the setup, model and dataset cells first."
    )

# 3. Load model checkpoint
try:
    checkpoint = torch.load(
        str(MODEL_PATH),
        map_location=device,
        weights_only=False
    )
except TypeError:
    checkpoint = torch.load(str(MODEL_PATH), map_location=device)

# Support either a checkpoint dictionary or plain model weights
if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    model.load_state_dict(checkpoint["model_state_dict"])
else:
    model.load_state_dict(checkpoint)

model.to(device)
model.eval()

if isinstance(checkpoint, dict) and "epoch" in checkpoint:
    print("Best epoch:", checkpoint["epoch"])

if isinstance(checkpoint, dict) and "val_mean_dice" in checkpoint:
    print("Saved validation Dice:", checkpoint["val_mean_dice"])

# 4. Select one validation scan
sample = val_ds[0]

image = sample["image"].unsqueeze(0).to(device)
true_mask = sample["label"].squeeze().cpu().numpy()

print("Image tensor shape:", tuple(image.shape))
print("Ground truth shape:", true_mask.shape)

# 5. Run sliding-window inference
with torch.no_grad():
    logits = sliding_window_inference(
        inputs=image,
        roi_size=PATCH_SIZE,
        sw_batch_size=1,
        predictor=model,
        overlap=0.25
    )

    prediction = torch.argmax(logits, dim=1).squeeze().cpu().numpy()

image_np = image.squeeze().cpu().numpy()

print("Predicted labels:", np.unique(prediction))
print("Ground truth labels:", np.unique(true_mask))

# 6. Find a slice containing predicted foreground.
# If no foreground is predicted, show the middle slice.
foreground_counts = np.sum(prediction > 0, axis=(0, 1))

if foreground_counts.max() > 0:
    slice_idx = int(np.argmax(foreground_counts))
else:
    slice_idx = image_np.shape[2] // 2

# 7. Display CT, actual mask and prediction
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(
    image_np[:, :, slice_idx].T,
    cmap="gray",
    origin="lower"
)
axes[0].set_title(f"Original CT - Slice {slice_idx}")

axes[1].imshow(
    true_mask[:, :, slice_idx].T,
    cmap="viridis",
    vmin=0,
    vmax=2,
    origin="lower",
    interpolation="nearest"
)
axes[1].set_title("Ground Truth Mask")

axes[2].imshow(
    image_np[:, :, slice_idx].T,
    cmap="gray",
    origin="lower"
)

pred_slice = prediction[:, :, slice_idx].T
pred_overlay = np.ma.masked_where(pred_slice == 0, pred_slice)

axes[2].imshow(
    pred_overlay,
    cmap="autumn",
    alpha=0.6,
    vmin=1,
    vmax=2,
    origin="lower",
    interpolation="nearest"
)
axes[2].set_title("U-Net Prediction Overlay")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

# 8. Report predicted voxel counts
print("\nPredicted voxel counts:")
for label, name in [(0, "Background"), (1, "Pancreas"), (2, "Cancer")]:
    print(f"{name} (class {label}): {int(np.sum(prediction == label))}")

print("\nVisualization complete.")
